In [0]:
CREATE OR REPLACE VIEW formula1ext.gold.v_driver_standing
AS

WITH driver_session_summary AS (
    SELECT
        r.season,
        d.driver_id,
        d.driver_name,
        d.nationality,
        COUNT_IF(r.session_type = 'RACE') AS race_starts,
        SUM(r.points) AS total_points,
        COUNT_IF(r.session_type = 'RACE' AND r.is_win) AS number_of_wins,
        COUNT_IF(r.session_type = 'RACE' AND r.is_podium) AS number_of_podiums
    FROM formula1ext.gold.fact_session_results r
    JOIN formula1ext.gold.dim_drivers d
      ON r.driver_id = d.driver_id
    GROUP BY
        r.season,
        d.driver_id,
        d.driver_name,
        d.nationality
),

driver_finish_sequence AS (
    SELECT
        season,
        driver_id,
        sort_array(
            collect_list(struct(final_position, round))
        ) AS finish_sequence
    FROM formula1ext.gold.fact_session_results
    WHERE session_type = 'RACE'
      AND final_position IS NOT NULL
      AND final_position > 0
      AND status <> 'Disqualified'
    GROUP BY season, driver_id
)

SELECT
    dss.season,
    dss.driver_id,
    dss.driver_name,
    dss.nationality,
    RANK() OVER (
        PARTITION BY dss.season
        ORDER BY
            dss.total_points DESC,
            dfs.finish_sequence ASC
    ) AS driver_standing,
    dss.race_starts,
    dss.total_points,
    dss.number_of_wins,
    dss.number_of_podiums
FROM driver_session_summary dss
LEFT JOIN driver_finish_sequence dfs
  ON dss.season = dfs.season
 AND dss.driver_id = dfs.driver_id;

In [0]:
SELECT * FROM formula1ext.gold.v_driver_standing WHERE season = 2025